In [ ]:
#Import necessary libraries

#%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import astropy
from astropy.table import Table, hstack
import scipy.stats
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.coordinates import Galactic
from astropy.coordinates import ICRS
import astropy.coordinates as apycord
import random
from cycler import cycler
import astropy.table

import re
import csv
import sys
import scipy
import pickle
import astropy
import pylab as p
import random as random
import numpy.random as rand
import astropy.io.fits as pyfits


from math import *
from math import cos, sin, pi
from numpy import *
from matplotlib import *
from scipy.interpolate import *
from astropy.table import join
from matplotlib.colors import LogNorm
import statistics as stat

# from scipy.integrate import trapz
# from scipy.integrate import quad

from astropy.table import QTable
from astropy.table import Table
import astropy.units as u
from astropy.coordinates import Angle

import zipfile as zf
import glob
from astropy.io import fits
import tarfile
import gzip
import shutil
import pandas as pd
from scipy.optimize import curve_fit

from concurrent.futures import ThreadPoolExecutor

import emcee
import corner

from PIL import Image
from IPython.display import display
import os

In [ ]:
#Read in the data covered by RUBIES (what we have spectra for) and by UNICORN (What we have photometric images for)

unicorn_area = Table.read("NIRSpec_Area_Photometry_EGS_UDS.fits") #len of 131720
rubies_sample = Table.read("RUBIES_PRISM_Auto_spec_z.fits") #len of 1951

In [ ]:
#The following redshift masks were deemed unnecessary and revoked

#Redshift mask on the file that contains all objects in the NIRSpec MSA pointings area, regardless of if they had a slit

unicorn_areaz = unicorn_area#[(unicorn_area.columns['ZPEAK']>2) & (unicorn_area.columns['ZPEAK']<7.1)] #len of 60798

#Redshift mask on my data as well, for these are the redshift bins that matter
rubies_samplez = rubies_sample#[(rubies_sample.columns['z_test']>2) & (rubies_sample.columns['z_test']<7.1)] #len of 1108

In [ ]:
rubies_samplez2 = rubies_samplez[(rubies_samplez.columns['MSA_ID']>0)]
rubies_samplez = rubies_samplez2 #cut out the negative MSA_ID values!

In [ ]:
#Display all RUBIES data
rubies_samplez

In [ ]:
#Display all UNICORN data
unicorn_areaz

In [ ]:
#crossmatching first between the two datasets using SkyCoord! Matches to closest coordinates within a limit

skycoord1 = SkyCoord(ra = unicorn_areaz['RA'], dec = unicorn_areaz['DEC'], unit = 'degree') #make skycoord1 the bigger set
skycoord2 = SkyCoord(ra = rubies_samplez['RA'], dec = rubies_samplez['DEC'], unit = 'degree')
idx, sep2d, sep3d = skycoord2.match_to_catalog_sky(skycoord1)

# rubies_samplez2=hstack([rubies_samplez,unicorn_areaz[idx]])
# rubies_samplez2=rubies_samplez2[sep.arcsec<0.25]

In [ ]:
#Now we make a mask so we only find matches for things that are actually (probably) the same
sep_mask = sep2d.arcsec < .25 #This is the value I have chosen 
matched_skycoord2 = skycoord2[sep_mask]
matched_idx_skycoord1 = idx[sep_mask]
matched_skycoord1 = skycoord1[matched_idx_skycoord1]
matched_sep = sep2d[sep_mask]


In [ ]:
#Print all the matches

print('All the close matches between the two catalogs')
print()
for sky1, sky2, sep in zip(matched_skycoord1, matched_skycoord2, matched_sep.arcsec):
    
    print(f'{sky1.ra:.6f}, {sky1.dec:.6f} -------- {sky2.ra:.6f}, {sky2.dec:.6f}')
print(len(matched_skycoord1))

In [ ]:
# Get unique matches based on indices of skycoord1
unique_idx, unique_indices = np.unique(matched_idx_skycoord1, return_index=True)

# Filter the arrays to remove duplicates
matched_skycoord1_unique = matched_skycoord1[unique_indices]
matched_skycoord2_unique = matched_skycoord2[unique_indices]
matched_sep_unique = matched_sep[unique_indices]

# Print the number of unique matches
print(len(matched_skycoord1_unique))
print(len(matched_skycoord2_unique))

In [ ]:
#Check how many did not match, verify this is a relatively small percentage (<10%)

no_match_ra=[]
no_match_dec=[]
match_ra=[]
match_dec=[]
for i in list(rubies_samplez['RA']):
    if i not in list(matched_skycoord2.ra.deg):
        no_match_ra.append(i)
    else:
        match_ra.append(i)
for i in list(rubies_samplez['DEC']):
    if i not in list(matched_skycoord2.dec.deg):
        no_match_dec.append(i)
    else:
        match_dec.append(i)
        
print(len(no_match_ra))
print(len(no_match_dec))
print(len(match_ra))
print(len(match_dec))

# Quick segway here into getting a magnitude column into the RUBIES data

In [ ]:
#Renaming rubies RA/DEC columns just to make sure I properly differentiate between the two datasets

rubies_samplez.rename_columns(["RA", "DEC"], ["RUBIES RA", "RUBIES DEC"])


In [ ]:
#Removed duplicate RUBIES matches using the unique idxs from above

filtered_rubies = rubies_samplez[np.isin(rubies_samplez["RUBIES RA"].astype(float), np.array(matched_skycoord2_unique.ra.deg))]
filtered_rubies

unique_ra, counts = np.unique(filtered_rubies["RUBIES RA"], return_counts=True)
duplicates = unique_ra[counts > 1]
#print(duplicates)
unique_ra, unique_indices = np.unique(filtered_rubies["RUBIES RA"], return_index=True)
filtered_rubies = filtered_rubies[unique_indices]


In [ ]:
#Only looking at the rubies objects with unique cross matches!
filtered_rubies.sort('RUBIES RA')
filtered_rubies

In [ ]:
#Only looking at the unicorn objects with unique matches!

filtered_unicorn = unicorn_areaz[np.isin(unicorn_areaz["RA"].astype(float), np.array(matched_skycoord1_unique.ra.deg))]
filtered_unicorn.sort('RA')
filtered_unicorn

In [ ]:
#Adding the magnitude column from unicorn into rubies!

filtered_rubies['Magnitude'] = (list(filtered_unicorn['Magnitude']))
filtered_rubies['UNICORN RA'] = list(filtered_unicorn['RA'])
filtered_rubies['UNICORN DEC'] = list(filtered_unicorn['DEC'])
filtered_rubies

In [ ]:
#Display the matched vs, unmatched objects in both the UDS and EGS fields (the 2 fields covered by RUBIES and UNICORN)

plt.figure()
plt.scatter(matched_skycoord2.ra.deg, matched_skycoord2.dec.deg, label = "Matched", alpha = .5)
#plt.scatter(no_match_ra, no_match_dec, color = 'red', label = "Unmatched", alpha = .5)
plt.title("NIRSpec Matches to Ultra Deep Survey Photometric Observations")
plt.xlabel("Right Ascension (Degrees)")
plt.ylabel("Declination (Degrees)")
plt.xlim(34.2,34.35)
plt.ylim(-5.35, -5.15)
plt.legend()
plt.show()


plt.figure()
plt.scatter(matched_skycoord2.ra.deg, matched_skycoord2.dec.deg, label = "Matched", alpha = .5)
#plt.scatter(no_match_ra, no_match_dec, color = 'red', label = "Unmatched", alpha = .5)
plt.title("NIRSpec Matches to Extended Groth Strip Photometric Observations")
plt.xlabel("Right Ascension (Degrees)")
plt.ylabel("Declination (Degrees)")
plt.legend()
plt.xlim(214.7, 215.2)
plt.ylim(52.75, 53.05)
plt.show()

In [ ]:
#Display the ratio of matched and unmatched objects with histograms (the two columns indicate the two fields)

plt.figure()
plt.hist(matched_skycoord2.ra.deg, label = "Matched", bins =np.arange(0,220,5))
plt.hist(no_match_ra, color = 'red', label = "Unmatched", bins = np.arange(0,220,5))
plt.xlabel("RA")
plt.legend()
plt.show()

plt.figure()
plt.hist(matched_skycoord2.dec.deg, label = "Matched", bins = np.arange(-10,60,5))
plt.hist(no_match_dec, color = 'red', label = "Unmatched", bins = np.arange(-10, 60, 5))
plt.legend()
plt.xlabel("DEC")
plt.show()

In [ ]:
#Guiding broad instructions if needed...

#First, CALCULATE magnitude so that we can later bin it...use flux through z-dep filters

#Then, bin up unicorn_areaz by redshifts

#Then, bin up those by magnitude

#Then, crossmatch using skycoords between the 1951 spectra and the 60798 total objects within my range.

#Once cross-matched, take the number of matches and divide by the total unicorn_areaz number in that redshift/magnitude bin

#Create completeness percentage plots for different redshifts as a function of magnitude

REMEBER: I don't need to bin up rubies_samplez, as I can just take the number of matches in each unicorn_areaz bin!

In [ ]:
#CALCULATE magnitude so that we can later bin it...use flux through z-dep filters

#The flux filter for which we measure flux from changes based on redshift as we want h-alpha in the proper range

filter_list = ['FLUX_F090W', 'FLUX_F115W', 'FLUX_F150W', 'FLUX_F200W', 'FLUX_F277W', 'FLUX_F356W', 'FLUX_F444W']
fluxes=[]
for num, z in enumerate(unicorn_areaz['ZPEAK']):
    if z >0 and z <= .54:
        flux_filter = filter_list[0]
    if z > .55 and z <=.98:
        flux_filter = filter_list[1]
    if z > .99 and z<= 1.6:
        flux_filter = filter_list[2]
    if z > 1.61 and z <= 2.52:
        flux_filter = filter_list[3]
    elif z>2.52 and z<=3.72:
        flux_filter = filter_list[4]
    elif z>3.72 and z<=4.96:
        flux_filter = filter_list[5]
    else:
        flux_filter = filter_list[6]
    flux = (unicorn_areaz[num][flux_filter]) #collecting the proper flux variable
    fluxes.append(flux)


In [ ]:
#Converting flux to magnitude, ignore nans(as replacing them with 0 doesnt work in magnitude space)
mag_vals=[]
for flux in fluxes:
    mag_vals.append(-2.5*np.log10(flux) + 31.4)
mag_vals

#add this magnitude column to the unicorn_areaz table
unicorn_areaz['Magnitude'] = mag_vals
unicorn_areaz['Fluxes'] = fluxes

In [ ]:
#Then, bin up unicorn_areaz by redshifts (6 total)

zs = np.arange(2.5,8.5)
unicorn_zbins=[]
#rubies_zbins=[]
for z in zs:
    binner = unicorn_areaz[np.abs(unicorn_areaz['ZPEAK']-z)<0.5]
    unicorn_zbins.append(binner)
#     binner = rubies_samplez[np.abs(rubies_samplez['z_test']-z)<0.5]
#     rubies_zbins.append(binner)
    


In [ ]:
#Then, bin up by magnitude within each redshift bin (13 each, 72 total)

mags = np.arange(16,40,1)
unicorn_magbins_z2 =[] #CENTERED at 2.5, and so on...
unicorn_magbins_z3 =[]
unicorn_magbins_z4 =[]
unicorn_magbins_z5 =[]
unicorn_magbins_z6 =[]
unicorn_magbins_z7 =[]
unicorn_magbins = [unicorn_magbins_z2, unicorn_magbins_z3, unicorn_magbins_z4, unicorn_magbins_z5, unicorn_magbins_z6,unicorn_magbins_z7]
for zbin in range(len(unicorn_magbins)):
    for mag in mags:
        binner = unicorn_zbins[zbin][np.abs(unicorn_zbins[zbin]['Magnitude'] - mag)<.5]
        unicorn_magbins[zbin].append(binner)

#Now, each of the 6 redshift bins have 13 magbins with a certain amount of objects in them

In [ ]:
#FInd the number of objects in each z+mag bin..these are the denominators
total_count=[]
for z in range(len(zs)):
    print()
    for mag in range(len(unicorn_magbins[z])):
        total_count.append(len(unicorn_magbins[z][mag]))

In [ ]:
#Now, we divide the number of rubies_samplez cross-matched objects in each bin..
print(len(unicorn_magbins[0]))


#Count the number of matches in each redshift and magnitude bin
match_count=[]
for z in range(len(zs)):
    for mag in range(len(unicorn_magbins[z])):
        matches=[]
        for i in unicorn_magbins[z][mag]['RA']:
            if i in list(matched_skycoord1.ra.deg):
                matches.append(i)
        match_count.append(len(matches))


In [ ]:
#Now, we can calculate the percentage of objects that we have NIRSpec spectra for in each redshift and magnitude bin

total_count = np.array(total_count)
match_count = np.array(match_count)

percentage = (match_count/total_count)*100

percentage

In [ ]:
#Chunk these into redshift bins for easier indexing

chunked = [percentage[i:i + 24] for i in range(0, len(percentage), 24)]

#Create color wheel to iterate through for redshift
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf', 'black']


In [ ]:
#Display redshift-separated observational completeness percentages
chunked

In [ ]:
#Plot the observational completeness percentage as a function of magnitude for each redshift bin

ax = plt.figure(figsize = (16,10), dpi = 250)
for i in range(len(chunked)):
    plt.plot(mags, chunked[i], color = colors[i], label = str(zs[i]-.5) + " < z < " + str(zs[i]+.5), linewidth = 3)
    plt.xlabel("Magnitude", fontsize = 35)
    plt.ylabel("Completeness Percentage", fontsize = 35)
    plt.title("Observational Completeness with Cross-Matching", fontsize = 40)
    plt.tick_params(axis = "both", labelsize = 20)
    plt.legend(fontsize = 30)
    plt.xlim(19,31)
plt.show()

In [ ]:
#go through 903 sources, get their completeness according to their mag+z, store as weight
chunked = np.array(chunked) #gives percents
print(1/chunked) #gives weights per bin
print(mags)

In [ ]:
#These are the weights we will multiply each object in the many tables above by

weights = (1/chunked)*100
weights

In [ ]:
#Now I have filtered rubies. I am going to calculate weights, and then add these weights into the original rubies_samplez
filt_rubies_zbins=[]
for z in zs:
    binner = filtered_rubies[np.abs(filtered_rubies['z_test']-z)<0.5]
    filt_rubies_zbins.append(binner)
    
    
filt_rubies_zbins[0]

filt_rubies_magbins = [[] for _ in range(6)]
for z in range(len(filt_rubies_zbins)):
    for mag in mags:
        binner = filt_rubies_zbins[z][np.abs(filt_rubies_zbins[z]['Magnitude'] - (mag+.5))<.5]
        filt_rubies_magbins[z].append(binner)



In [ ]:
#This is now a list of lists of tables. there are 6 lists for each redshift range, and each of those has 24 mag lists
#each of those 24 mag lists is an Astropy Table with all the required columns. now I need to match these to chunked bins above
z_ind = 0
keeps=[]
for z in filt_rubies_magbins:
    mag_ind = 0
    for mag in z:
        display(mag)
        print(weights[z_ind][mag_ind])
        if len(mag) >0:
            mag['Weight'] = weights[z_ind][mag_ind]
            keeps.append(mag)
        mag_ind+=1
    z_ind+=1
        

In [ ]:
#Now, just need to recombine all the minitables with weights into a big one again
#Now, each source that was matched between RUBIES and UNICORN as a weight depending on the observational completeness percentage
#of an object with its specific redshift and magnitude(which is calculated through a redshift-dependent filter)
#This for 900+ objects, which include most of the 504 objects in my actual sample. I will multiply each object by its weight
#when I plot the corrected LFs

from astropy.table import vstack

filtered_rubies_weights = vstack(keeps)
filtered_rubies_weights

In [ ]:
#Removing the nan and inf values (as this would mean there are no UNICORN or RUBIES/both object respectively, in that bin)

weight_mask = ~np.isnan(filtered_rubies_weights["Weight"]) & ~np.isinf(filtered_rubies_weights["Weight"])

rubies_weights = filtered_rubies_weights[weight_mask]
rubies_weights

In [ ]:
rubies_weights.show_in_browser()